In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")



In [2]:
##CALLING THE MODEL

from langchain_groq import ChatGroq


chat_groq = ChatGroq(model="llama-3.1-8b-instant", api_key=GROQ_API_KEY)

d:\shree\project_files\samples\langchain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#INVOKE THE MESSAGES ON THE MODEL
from langchain_core.messages import SystemMessage, HumanMessage

chat_groq.invoke([
    HumanMessage(content="Hi I am Shree, I am AI learner")])

AIMessage(content="Nice to meet you, Shree.  I'm happy to help you with any questions or topics you'd like to learn about. What's on your mind? Are you interested in a specific subject, such as AI, programming, or something else? Or do you want to explore a particular topic or concept? I'm here to help you learn and grow.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 45, 'total_tokens': 119, 'completion_time': 0.117943461, 'completion_tokens_details': None, 'prompt_time': 0.002189466, 'prompt_tokens_details': None, 'queue_time': 0.051135461, 'total_time': 0.120132927}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019de947-1bed-7bf0-ba14-41bbd82410c4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 45, 'output_tokens': 74, 'total_tokens': 119})

Trim message 

In [5]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, trim_messages

trimmer = trim_messages(max_tokens=10, strategy="last",token_counter=chat_groq,include_system=True,allow_partial=False,start_on="human")

In [6]:
## USING PROMPT TEMPLATES

from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content="You are a helpful assistant"),
    MessagesPlaceholder(variable_name="messages")
    ])

chain = prompt | chat_groq    

In [7]:
## CHAT MESSAGE HISTORY

from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

stored={}

#create a session to differentioniate between different sessions
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in stored:
        stored[session_id] = ChatMessageHistory()
    return stored[session_id]

with_chat_history = RunnableWithMessageHistory(chain,get_session_history,input_message_key="messages")    

In [8]:
def get_config(session_id):
    return {"configurable":{"session_id": session_id}}

def ask_bot(input,session_id):
    Config = get_config(session_id)
    response = with_chat_history.invoke({"messages":[HumanMessage(content=input)]},Config)
    return response.content

In [9]:
ask_bot("Hi I am Alpha","session1")

'Nice to meet you, Alpha. Is there something I can help you with today?'

In [12]:
ask_bot("What is my name?","session1")

'Your name is Alpha.'

In [13]:
ask_bot("The astronauts of NASA’s Artemis II mission have offered a vivid, human account of their journey around the Moon, describing Earth as a “tiny” speck in space and their return as a fiery plunge through the atmosphere during a recent appearance on The Tonight Show Starring, Jimmy Fallon.Commander Reid Wiseman, pilot Victor Glover, and mission specialists Christina Koch and Jeremy Hansen spoke about the emotional and physical extremes of travelling farther from Earth than any humans in over half a century. At the mission’s peak distance, Earth appeared small and fragile—an image the crew said left a lasting impression about the planet’s vulnerability","session3")

'The astronauts of NASA\'s Artemis II mission have shared their incredible experience with the world. Their journey around the Moon has given them a unique perspective on our planet, and their description of Earth as a "tiny" speck in space resonates with the fragility of our existence.\n\nThe fact that they saw Earth in a new light, quite literally, highlights the reality of our planet\'s smallness in the grand scheme of the universe. It\'s a humbling experience that can evoke a sense of awe and appreciation for the beauty and vulnerability of our home.\n\nAs Commander Reid Wiseman, pilot Victor Glover, and mission specialists Christina Koch and Jeremy Hansen shared their story on The Tonight Show Starring Jimmy Fallon, they also spoke about the emotional and physical extremes they faced during their journey. Traveling farther from Earth than any humans in over half a century can\'t be easy, and it\'s admirable that they were able to share their experiences with the public.\n\nTheir r

In [14]:
ask_bot("What is was mission name?","session3")

"The mission of the astronauts who spoke about their journey around the Moon on The Tonight Show Starring Jimmy Fallon is called Artemis II. \n\nHowever, since the information you initially provided mentions the astronauts' appearance on The Tonight Show Starring Jimmy Fallon, I believe you might be referring to the Artemis II mission astronauts who appeared on the show.\n\nTo confirm, the astronauts mentioned in the text are from the Artemis II mission, which is the first crewed mission of the Artemis program."